In [85]:
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Literal
from dotenv import load_dotenv, find_dotenv
import os
from pathlib import Path
import json
import pandas as pd
import numpy as np
import torch.nn.functional as F
import time
import base64
import mimetypes


In [91]:
dotenv_path = find_dotenv()
print("dotenv path:", dotenv_path)

load_dotenv(dotenv_path, override=True)

API_KEY = os.getenv("OPENAI_API_KEY")

dotenv path: d:\Enrichment Real\sarcasm-detection\.env


In [61]:
MODEL_NAME = "gpt-4.1-mini"
DATA_DIR  = Path("../Dataset/JSON")
OUTPUT_DIR = Path("../Output")
IMAGE_DATA_DIR = Path("../Dataset/dataset_image")

In [92]:
client = OpenAI(api_key=API_KEY)

In [24]:
with open(DATA_DIR / "test.json", encoding="utf-8") as f:
    test_data = json.load(f)

with open(DATA_DIR / "valid.json", encoding="utf-8") as f:
    val_data = json.load(f)

with open(DATA_DIR / "train.json", encoding="utf-8") as f:
    train_data = json.load(f)

In [28]:
test_df = pd.DataFrame(test_data)
val_df = pd.DataFrame(val_data)
train_df = pd.DataFrame(train_data)

In [36]:
train_metadata = pd.read_csv("../Output/train_metadata.csv")
metadata_df = train_metadata.copy().reset_index(drop=True)

metadata_df.head(5)

,image_path,text,label,image_id
0,..\Dataset\dataset_image\840006160660983809.jpg,<user> thanks for showing up for our appointme...,1,840006160660983809
1,..\Dataset\dataset_image\908913372199915520.jpg,haha .,1,908913372199915520
2,..\Dataset\dataset_image\916496521406726145.jpg,i love waiting <num> min for a cab - such shor...,1,916496521406726145
3,..\Dataset\dataset_image\916364004129304576.jpg,22 super funny quotes <user>,1,916364004129304576
4,..\Dataset\dataset_image\853866052589154304.jpg,goog morning,1,853866052589154304


In [37]:
metadata_df["row_id"] = np.arange(len(metadata_df))
metadata_df.head(5)

,image_path,text,label,image_id,row_id
0,..\Dataset\dataset_image\840006160660983809.jpg,<user> thanks for showing up for our appointme...,1,840006160660983809,0
1,..\Dataset\dataset_image\908913372199915520.jpg,haha .,1,908913372199915520,1
2,..\Dataset\dataset_image\916496521406726145.jpg,i love waiting <num> min for a cab - such shor...,1,916496521406726145,2
3,..\Dataset\dataset_image\916364004129304576.jpg,22 super funny quotes <user>,1,916364004129304576,3
4,..\Dataset\dataset_image\853866052589154304.jpg,goog morning,1,853866052589154304,4


In [57]:
def make_balanced_subset(df, n_each, random_state=42):
    sarcasm_df = df[df["label"] == 1]
    non_sarcasm_df = df[df["label"] == 0]

    if len(sarcasm_df) < n_each:
        raise ValueError(f"Sarcasm rows only {len(sarcasm_df)}, need {n_each}")

    if len(non_sarcasm_df) < n_each:
        raise ValueError(f"Non-sarcasm rows only {len(non_sarcasm_df)}, need {n_each}")

    sampled_sarcasm = sarcasm_df.sample(n=n_each, random_state=random_state)
    sampled_non_sarcasm = non_sarcasm_df.sample(n=n_each, random_state=random_state)

    out = pd.concat([sampled_sarcasm, sampled_non_sarcasm], ignore_index=True)
    out = out.sample(frac=1, random_state=random_state).reset_index(drop=True)

    return out

In [ ]:
smoke_df = make_balanced_subset(metadata_df, n_each=10, random_state=42)

smoke_df.to_csv("../Output/smoke-tesr-raw.csv", index=False)
rationale_df = make_balanced_subset(metadata_df, n_each=1000, random_state=67)

print("Smoke label count:")
print(smoke_df["label"].value_counts())

print("Main label count:")
print(rationale_df["label"].value_counts())

Smoke label count:
label
1    10
0    10
Name: count, dtype: int64
Main label count:
label
0    1000
1    1000
Name: count, dtype: int64


In [ ]:
smoke_df.to_csv("../Output/smoke-test-raw.csv", index=False)

In [107]:
rationale_df.to_csv("../Output/main-dataset-raw.csv", index=False)

In [42]:
train_embv = np.load(OUTPUT_DIR / "train_embv.npy").astype("float32")
train_embt = np.load(OUTPUT_DIR / "train_embt.npy").astype("float32")

In [43]:
def l2_normalize_np(x, axis=-1, eps=1e-12):
    return x / (np.linalg.norm(x, axis=axis, keepdims=True) + eps)

In [44]:
def fuse_embeddings_np(embv, embt, alpha=0.5):
    embv = np.asarray(embv).astype("float32")
    embt = np.asarray(embt).astype("float32")

    fused = alpha * embv + (1 - alpha) * embt
    fused = l2_normalize_np(fused, axis=-1)

    return fused.astype("float32")

In [45]:
def fuse_embeddings_torch(embv, embt, alpha=0.5):
    fused = alpha * embv + (1 - alpha) * embt
    fused = F.normalize(fused, p=2, dim=-1)

    return fused

In [47]:
train_fused_embedding = fuse_embeddings_np(
    train_embv,
    train_embt,
).squeeze(1)

train_fused_embedding.shape

print(train_fused_embedding.shape)
assert train_fused_embedding.shape[0] == len(metadata_df)

(19816, 768)


In [75]:
def get_image_path(row):
    if "image_id" in row and pd.notna(row["image_id"]):
        return IMAGE_DATA_DIR / f'{str(row["image_id"])}.jpg'

    raw_path = str(row["image_path"]).replace("\\", "/")
    filename = Path(raw_path).name

    return IMAGE_DATA_DIR / filename


def image_to_data_url(image_path):
    image_path = Path(image_path)

    if not image_path.exists():
        raise FileNotFoundError(f"Image not found: {image_path}")

    mime_type, _ = mimetypes.guess_type(str(image_path))

    if mime_type is None:
        suffix = image_path.suffix.lower()

        if suffix in [".jpg", ".jpeg"]:
            mime_type = "image/jpeg"
        elif suffix == ".png":
            mime_type = "image/png"
        elif suffix == ".webp":
            mime_type = "image/webp"
        else:
            mime_type = "image/jpeg"

    with open(image_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("utf-8")

    return f"data:{mime_type};base64,{b64}"

In [55]:
def to_numpy_query(query_embedding):
    if hasattr(query_embedding, "detach"):
        query_vec = query_embedding.squeeze(0).detach().cpu().numpy()
    else:
        query_vec = np.asarray(query_embedding).squeeze()

    return query_vec.astype("float32")


def retrieve_top_k_by_label(
    query_embedding,
    db_embeddings,
    metadata_df,
    top_k_each=2,
    sarcasm_label=1,
    non_sarcasm_label=0,
    exclude_row_id=None,
    exclude_image_id=None
):
    query_vec = to_numpy_query(query_embedding)

    db_embeddings = np.asarray(db_embeddings).astype("float32")
    scores = db_embeddings @ query_vec

    if exclude_row_id is not None:
        scores[int(exclude_row_id)] = -np.inf

    if exclude_image_id is not None:
        mask = metadata_df["image_id"].astype(str) == str(exclude_image_id)
        scores[mask.values] = -np.inf

    df = metadata_df.copy()
    df["score"] = scores

    sarcasm_results = (
        df[df["label"] == sarcasm_label]
        .sort_values("score", ascending=False)
        .head(top_k_each)
        .copy()
    )

    non_sarcasm_results = (
        df[df["label"] == non_sarcasm_label]
        .sort_values("score", ascending=False)
        .head(top_k_each)
        .copy()
    )

    return sarcasm_results, non_sarcasm_results

In [64]:
def safe_text(x):
    if pd.isna(x):
        return ""
    return str(x).replace("\n", " ").strip()


def label_name(label):
    return "sarcasm" if int(label) == 1 else "not sarcasm"


def format_retrieved_examples(df, title):
    lines = [f"{title}:"]

    if df is None or len(df) == 0:
        lines.append("- No retrieved examples available.")
        return "\n".join(lines)

    for i, (_, r) in enumerate(df.iterrows(), start=1):
        lines.append(
            f"{i}. Text: {safe_text(r.get('text', ''))}\n"
            f"   Label: {label_name(r.get('label', 0))}\n"
            f"   Similarity score: {float(r.get('score', 0.0)):.4f}"
        )

    return "\n".join(lines)

In [62]:
class SingleRationaleOutput(BaseModel):
    rationale: str = Field(
        description="A concise rationale from the requested stance only."
    )

In [65]:
SYSTEM_PROMPT = """
You are a careful annotator for multimodal sarcasm reasoning.

You are given:
1. A query post consisting of text and an image.
2. Retrieved examples that already have labels, such as sarcasm or non-sarcasm.
3. A requested standpoint, either sarcastic or non-sarcastic.

The query label is unknown and must not be assumed.
The retrieved example labels are known and should be used only as supporting references for comparison.

Generate a rationale from the requested standpoint only.

Do not make a final classification decision.
Do not predict the query label.
Do not mention a gold label for the query.
Do not say whether the requested standpoint is correct or incorrect.
""".strip()

In [66]:
def build_stance_prompt(query_row, sarcasm_res, non_sarcasm_res, stance):
    query_text = safe_text(query_row["text"])

    prompt = f"""
Given a query post that consists of text and an image, please give a rationale of why the query post may be {stance}.

Query text:
{query_text}

Retrieved labeled examples for comparison:

{format_retrieved_examples(sarcasm_res, "Top 2 retrieved examples labeled sarcasm")}

{format_retrieved_examples(non_sarcasm_res, "Top 2 retrieved examples labeled non-sarcasm")}

Task:
Write only one rationale from the standpoint that the query post may be {stance}.

Rules:
- The query label is not provided.
- The retrieved example labels are provided and may be used as comparison evidence.
- Do not make a final classification decision.
- Do not predict the query label.
- Do not mention the true/gold label of the query.
- Use the query text and visible image context as the main evidence.
- Use retrieved labeled examples only as supporting references.
- Do not simply copy the retrieved examples.
- Keep the rationale concise, around 1-3 sentences.
""".strip()

    return prompt

In [ ]:
def build_stance_prompt(query_row, sarcasm_res, non_sarcasm_res, stance):
    query_text = safe_text(query_row["text"])

    prompt = f"""
    Given a post that consists of a text and an image, please give a rationale of why the post is {stance}.

    Query text:
    {query_text}

    Retrieved evidence for comparison:

    {format_retrieved_examples(sarcasm_res, "Top 2 retrieved examples labeled sarcasm")}

    {format_retrieved_examples(non_sarcasm_res, "Top 2 retrieved examples labeled not sarcasm")}

    Task:
    Write the rationale only from the standpoint that the query post is {stance}.

    Rules:
    - Do not make a final classification decision.
    - Do not mention the true/gold label.
    - Do not say "the label is".
    - Use the query text and visible image context as the main evidence.
    - Use the retrieved examples only as supporting comparison.
    - Do not simply copy the retrieved examples.
    - Keep the rationale concise, around 1-3 sentences.
    """.strip()

    return prompt

In [69]:
def format_retrieved_examples(df, title):
    lines = [f"{title}:"]

    if df is None or len(df) == 0:
        lines.append("- No retrieved examples available.")
        return "\n".join(lines)

    for i, (_, r) in enumerate(df.iterrows(), start=1):
        lines.append(
            f"{i}. Text: {safe_text(r.get('text', ''))}\n"
            f"   Example label: {label_name(r.get('label', 0))}\n"
            f"   Similarity score: {float(r.get('score', 0.0)):.4f}"
        )

    return "\n".join(lines)

In [72]:
def generate_single_stance_rationale(
    row,
    sarcasm_res,
    non_sarcasm_res,
    stance,
    model=MODEL_NAME,
    include_image=True,
    image_detail="low"
):
    prompt = build_stance_prompt(
        query_row=row,
        sarcasm_res=sarcasm_res,
        non_sarcasm_res=non_sarcasm_res,
        stance=stance
    )

    content = [
        {
            "type": "input_text",
            "text": prompt
        }
    ]

    if include_image:
        image_path = get_image_path(row)
        content.append(
            {
                "type": "input_image",
                "image_url": image_to_data_url(image_path),
                "detail": image_detail
            }
        )

    response = client.responses.parse(
        model=model,
        instructions=SYSTEM_PROMPT,
        input=[
            {
                "role": "user",
                "content": content
            }
        ],
        text_format=SingleRationaleOutput,
    )

    return response.output_parsed.rationale

In [73]:
def generate_rationale_for_row(
    row,
    metadata_df,
    db_embeddings,
    top_k_each=2,
    model=MODEL_NAME,
    include_image=True,
    image_detail="low",
    sleep_s=0.0
):
    row_id = int(row["row_id"])

    # Query embedding langsung dari fused train embedding
    query_embedding = db_embeddings[row_id].reshape(1, -1)

    # Retrieve top-k sarcasm dan top-k non-sarcasm
    sarcasm_res, non_sarcasm_res = retrieve_top_k_by_label(
        query_embedding=query_embedding,
        db_embeddings=db_embeddings,
        metadata_df=metadata_df,
        top_k_each=top_k_each,
        sarcasm_label=1,
        non_sarcasm_label=0,
        exclude_row_id=row_id,
        exclude_image_id=row["image_id"]
    )

    # Generate rationale dari stance sarcastic
    sarcasm_rationale = generate_single_stance_rationale(
        row=row,
        sarcasm_res=sarcasm_res,
        non_sarcasm_res=non_sarcasm_res,
        stance="sarcastic",
        model=model,
        include_image=include_image,
        image_detail=image_detail
    )

    if sleep_s > 0:
        time.sleep(sleep_s)

    not_sarcasm_rationale = generate_single_stance_rationale(
        row=row,
        sarcasm_res=sarcasm_res,
        non_sarcasm_res=non_sarcasm_res,
        stance="non-sarcastic",
        model=model,
        include_image=include_image,
        image_detail=image_detail
    )

    if sleep_s > 0:
        time.sleep(sleep_s)

    return {
        "row_id": row_id,
        "image_id": row["image_id"],
        "image_path": row.get("image_path", None),
        "text": row["text"],

        "true_label": int(row["label"]),

        "sarcasm_rationale": sarcasm_rationale,
        "not_sarcasm_rationale": not_sarcasm_rationale,

        "top_sarcasm_ids": sarcasm_res["image_id"].astype(str).tolist(),
        "top_sarcasm_texts": sarcasm_res["text"].astype(str).tolist(),
        "top_sarcasm_scores": sarcasm_res["score"].astype(float).tolist(),

        "top_non_sarcasm_ids": non_sarcasm_res["image_id"].astype(str).tolist(),
        "top_non_sarcasm_texts": non_sarcasm_res["text"].astype(str).tolist(),
        "top_non_sarcasm_scores": non_sarcasm_res["score"].astype(float).tolist(),
    }

In [ ]:
smoke_results = []

for i, (_, row) in enumerate(smoke_df.iterrows(), start=1):
    try:
        result = generate_rationale_for_row(
            row=row,
            metadata_df=metadata_df,
            db_embeddings=train_fused_embedding,
            top_k_each=2,
            model=MODEL_NAME,
            include_image=True,
            image_detail="low",
            sleep_s=0.1
        )

        smoke_results.append(result)

        print(
            f"[OK] {i}/{len(smoke_df)} "
            f"row_id={row['row_id']} "
            f"image_id={row['image_id']} "
            f"label={row['label']}"
        )

    except Exception as e:
        print(
            f"[ERROR] {i}/{len(smoke_df)} "
            f"row_id={row.get('row_id', None)} "
            f"image_id={row.get('image_id', None)}"
        )
        print(e)

        smoke_results.append({
            "row_id": row.get("row_id", None),
            "image_id": row.get("image_id", None),
            "image_path": row.get("image_path", None),
            "text": row.get("text", None),
            "true_label": row.get("label", None),
            "error": str(e)
        })

smoke_result_df = pd.DataFrame(smoke_results)
smoke_result_df


[OK] 1/20 row_id=15021 image_id=869558893386616832 label=1
[OK] 2/20 row_id=2394 image_id=822224089351786496 label=0
[OK] 3/20 row_id=18824 image_id=820419717441540099 label=0
[OK] 4/20 row_id=10907 image_id=906553309976084480 label=1
[OK] 5/20 row_id=15306 image_id=846025545724956676 label=1
[OK] 6/20 row_id=15248 image_id=716067194799521796 label=1
[OK] 7/20 row_id=13534 image_id=821868104284246017 label=0
[OK] 8/20 row_id=15806 image_id=714920275444412416 label=1
[OK] 9/20 row_id=11937 image_id=822953000314531841 label=0
[OK] 10/20 row_id=14389 image_id=821504098872606723 label=0
[OK] 11/20 row_id=8779 image_id=819326696063168513 label=0
[OK] 12/20 row_id=2638 image_id=937776053782753283 label=1
[OK] 13/20 row_id=12949 image_id=910968533928022016 label=1
[OK] 14/20 row_id=6802 image_id=818603167646699520 label=0
[OK] 15/20 row_id=3023 image_id=849851250917089280 label=1
[OK] 16/20 row_id=9317 image_id=819688762166968322 label=0
[OK] 17/20 row_id=10694 image_id=920748848087044096 lab

,row_id,image_id,image_path,text,true_label,sarcasm_rationale,not_sarcasm_rationale,top_sarcasm_ids,top_sarcasm_texts,top_sarcasm_scores,top_non_sarcasm_ids,top_non_sarcasm_texts,top_non_sarcasm_scores
0,15021,869558893386616832,..\Dataset\dataset_image\869558893386616832.jpg,the only way out of getting tracked by your en...,1,The post pairs a humorous image of unusual sho...,The post straightforwardly presents a humorous...,"[797811315532304384, 902001115532636160]","[that is just evil .., it will work]","[0.7111637592315674, 0.7025550603866577]","[820419359625637889, 822590809396703233]","[i knew it ..., lmao]","[0.6848856210708618, 0.6703442335128784]"
1,2394,822224089351786496,..\Dataset\dataset_image\822224089351786496.jpg,def wearing this to the women 's march in dc .,0,"The phrase ""def wearing this to the women 's m...",The post straightforwardly expresses the inten...,"[822949969162342401, 817778969474961409]",[don 't forget : white women voted for trump ....,"[0.6815959811210632, 0.6756000518798828]","[823315348107632640, 822950696026652675]","[nasty women unite !, so great seeing <user> a...","[0.6987298130989075, 0.6875726580619812]"
2,18824,820419717441540099,..\Dataset\dataset_image\820419717441540099.jpg,i 'm in love with the shape of you,0,"The text ""I'm in love with the shape of you"" p...","The query text ""i'm in love with the shape of ...","[930503214302093314, 834501103261724677]","[might delete this but felt cute today, wow lo...","[0.7494580149650574, 0.741487979888916]","[818607685113380865, 820056122400739328]","[im in love with the shape of you, hello]","[0.8338158130645752, 0.795893669128418]"
3,10907,906553309976084480,..\Dataset\dataset_image\906553309976084480.jpg,22 of the funniest quotes you 'll read <user>,1,"The post's juxtaposition of the phrase ""seized...",The post expresses straightforward frustration...,"[904519715640360961, 904772598080323588]",[22 of the funniest quotes you 'll read <user>...,"[1.000000238418579, 0.8849201798439026]","[820410608038318080, 822228287380459521]","[great advice ! !, true ! !]","[0.7297735214233398, 0.7247034907341003]"
4,15306,846025545724956676,..\Dataset\dataset_image\846025545724956676.jpg,<user> <user> it looks like there is a lack of...,1,The post sarcastically highlights the irony be...,The post's statement about a lack of leadershi...,"[824740167105720324, 818593969080766468]",[i 'm glad there 's someone here to help . tha...,"[0.6957891583442688, 0.694485604763031]","[820052922167861248, 819324917292396544]",[the law of the lid : leadership ability deter...,"[0.7315047979354858, 0.7257100343704224]"
5,15248,716067194799521796,..\Dataset\dataset_image\716067194799521796.jpg,most # brutal # customerservice # call i 've h...,1,The post sarcastically describes a customer se...,The query text straightforwardly describes a l...,"[816721159614058499, 875474667074715651]",[my current time on your holine <user> great c...,"[0.8395407199859619, 0.8312991857528687]","[817517691862777856, 820056592074862592]","[<user> what gives br0 ?, current mood]","[0.7053813338279724, 0.7001335024833679]"
6,13534,821868104284246017,..\Dataset\dataset_image\821868104284246017.jpg,squad goals,0,"The phrase ""squad goals"" is used sarcastically...","The phrase ""squad goals"" combined with the ima...","[791426760910991360, 823315658620465154]","[squad goals . <user> <user> <user>, this is w...","[0.674962043762207, 0.6630423665046692]","[818244795772481536, 822226408235614208]","[ultimate squad goals, when you all decide to ...","[0.7675034403800964, 0.7674597501754761]"
7,15806,714920275444412416,..\Dataset\dataset_image\714920275444412416.jpg,good thing islam is peaceful can you imagine h...,1,"The query text uses the phrase ""good thing Isl...",The query text expresses a straightforward sta...,"[819695309660061696, 815946368195919872]",[guess who is for unlimited muslim immigration...,"[0.8794822692871094, 0.8721713423728943]","[817518261726224384, 819692888573571072]","[i

In [96]:
train_df.head(5)

,image_id,text,label
0,840006160660983809,<user> thanks for showing up for our appointme...,1
1,908913372199915520,haha .,1
2,916496521406726145,i love waiting <num> min for a cab - such shor...,1
3,916364004129304576,22 super funny quotes <user>,1
4,853866052589154304,goog morning,1


In [103]:

train_df[train_df["label"] == 1]

,image_id,text,label
0,840006160660983809,<user> thanks for showing up for our appointme...,1
1,908913372199915520,haha .,1
2,916496521406726145,i love waiting <num> min for a cab - such shor...,1
3,916364004129304576,22 super funny quotes <user>,1
4,853866052589154304,goog morning,1
...,...,...,...
19744,822951233573023744,me : save money in 2017 bassnectar me : book p...,1
19776,820051335039119360,"joe : can you help me , i can 't reach the cla...",1
19785,820416096633233414,you : should i park the car in the garage or i...,1
19790,820050768791224320,rain drop drop top i 'm gonna hit you upside t...,1


In [94]:
output_path = "../Output/smoke_test_llm_result.csv"
smoke_result_df.to_csv(output_path, index=False)

In [106]:
output_path = "../Output/few-shot-main-result.csv"

main_results = []
save_every = 25

for i, (_, row) in enumerate(rationale_df.iterrows(), start=1):
    try:
        result = generate_rationale_for_row(
            row=row,
            metadata_df=metadata_df,
            db_embeddings=train_fused_embedding,
            top_k_each=2,
            model=MODEL_NAME,
            include_image=True,
            image_detail="low",
            sleep_s=0.1
        )

        main_results.append(result)

        print(
            f"[OK] {i}/{len(rationale_df)} "
            f"row_id={row['row_id']} "
            f"image_id={row['image_id']} "
            f"label={row['label']}"
        )

    except Exception as e:
        print(
            f"[ERROR] {i}/{len(rationale_df)} "
            f"row_id={row.get('row_id', None)} "
            f"image_id={row.get('image_id', None)}"
        )
        print(e)

        main_results.append({
            "row_id": row.get("row_id", None),
            "image_id": row.get("image_id", None),
            "image_path": row.get("image_path", None),
            "text": row.get("text", None),
            "true_label": row.get("label", None),
            "error": str(e)
        })

    if i % save_every == 0:
        main_result_df = pd.DataFrame(main_results)
        main_result_df.to_csv(output_path, index=False)
        print(f"[SAVED] checkpoint {i}/{len(rationale_df)} to {output_path}")

main_result_df = pd.DataFrame(main_results)
main_result_df.to_csv(output_path, index=False)

print(f"[DONE] Saved final result to {output_path}")

main_result_df

[OK] 1/2000 row_id=4181 image_id=819692441481711617 label=0
[OK] 2/2000 row_id=5418 image_id=923000568997539840 label=1
[OK] 3/2000 row_id=1850 image_id=818238963156680705 label=0
[OK] 4/2000 row_id=5650 image_id=877881532404379648 label=1
[OK] 5/2000 row_id=9656 image_id=822594021897961472 label=0
[OK] 6/2000 row_id=19156 image_id=820053476088696833 label=0
[OK] 7/2000 row_id=903 image_id=916681911837261824 label=1
[OK] 8/2000 row_id=9364 image_id=823312295124226050 label=0
[OK] 9/2000 row_id=12863 image_id=822806684829708288 label=1
[OK] 10/2000 row_id=1207 image_id=822226632807170048 label=0
[OK] 11/2000 row_id=3726 image_id=821504936286584834 label=0
[OK] 12/2000 row_id=10802 image_id=875130409205420032 label=1
[OK] 13/2000 row_id=19674 image_id=819330300333301761 label=0
[OK] 14/2000 row_id=7452 image_id=872983458691338241 label=1
[OK] 15/2000 row_id=7751 image_id=779159315944837120 label=1
[OK] 16/2000 row_id=5943 image_id=825549849290432512 label=1
[OK] 17/2000 row_id=8292 image

,row_id,image_id,image_path,text,true_label,sarcasm_rationale,not_sarcasm_rationale,top_sarcasm_ids,top_sarcasm_texts,top_sarcasm_scores,top_non_sarcasm_ids,top_non_sarcasm_texts,top_non_sarcasm_scores,error
0,4181,819692441481711617,..\Dataset\dataset_image\819692441481711617.jpg,i 'm rethinking everything now,0,"The phrase ""I'm rethinking everything now"" pai...",The post expresses a thoughtful and sincere re...,"[914592921814433794, 921202878571864065]","[you ever think about that ? no , you only thi...","[0.744874119758606, 0.7275177240371704]","[819694428277379074, 818607947706200064]","[i 'm rethinking everything now, wayment !]","[1.0000001192092896, 0.9519281983375549]",NaN
1,5418,923000568997539840,..\Dataset\dataset_image\923000568997539840.jpg,# truth about,1,The post uses exaggeration and ironic phrasing...,The text presents a straightforward sentiment ...,"[712358912264110080, 694584307655012358]","[true ., # truth]","[0.7744077444076538, 0.7734344601631165]","[822225305779785730, 819694201831096321]","[exactly !, but really though]","[0.7239524722099304, 0.7229138016700745]",NaN
2,1850,818238963156680705,..\Dataset\dataset_image\818238963156680705.jpg,took a nap and slept through a pretty sunset i...,0,The post sarcastically contrasts the disappoin...,The post expresses a straightforward sentiment...,"[839491693049171968, 698338846287785984]","[haha when you know people care ! .... jk, nob...","[0.6213423013687134, 0.5821914672851562]","[822227785137655809, 822585733273808899]",[first thing i see on snap is food <user> <use...,"[0.7123841643333435, 0.6349201202392578]",NaN
3,5650,877881532404379648,..\Dataset\dataset_image\877881532404379648.jpg,- akhileshyadav is struggling for a govt job r...,1,The post sarcastically suggests that Akhilesh ...,The post straightforwardly describes Akhilesh ...,"[880436067270356993, 877715414763130880]",[: a govt employee got delighted after rescuin...,"[0.7288748025894165, 0.7151439189910889]","[819332300399779841, 818238095594291200]",[queen of flowers queen of farmers priyanka ga...,"[0.5446099638938904, 0.5340462923049927]",NaN
4,9656,822594021897961472,..\Dataset\dataset_image\822594021897961472.jpg,quand jma fait signer des cracks > > > > >,0,The post text suggests a boastful claim about ...,The post expresses pride and confidence in hav...,"[686926157288202241, 902261356774207489]",[im in the process of sealing a £ 12m switch t...,"[0.5847525596618652, 0.5751825571060181]","[819688464740450304, 822587074209583104]","[i have officially completed life ., brace you...","[0.6404160261154175, 0.6285971999168396]",NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,2586,845261518119849985,..\Dataset\dataset_image\845261518119849985.jpg,fridaymorning at the bus stop the 5th day of s...,1,"The text describes a warm, vibrant spring morn...",The text describes a pleasant spring morning w...,"[843821536234655744, 717110905683767297]",[happy spring ! loving all the blossoming flow...,"[0.6721720695495605, 0.6278952956199646]","[819323350380003328, 818242546983800834]","[good morning world, the end of the beginning ...","[0.5985373854637146, 0.596052348613739]",NaN
1996,16040,822592036201254912,..\Dataset\dataset_image\822592036201254912.jpg,how many retweets can we get for the first lad...,0,"The query's phrasing, asking ""how many retweet...",The query text straightforwardly seeks to gath...,"[823315156335783936, 822586355465285639]",[calling it . melania will bring down the trum...,"[0.8372665047645569, 0.8110149502754211]","[822593370602278914, 822222913151664134]",[the president & first lady of the united stat...,"[0.8336414694786072, 0.8208781480789185]",NaN
1997,14243,818605078378647554,..\Dataset\dataset_image\818605078378647554.jpg,well deserved contact extension .,0,"The phrase ""well deserved contact extension"" p...","The phrase ""well deserved contract extension"" ...","[818593969080766468, 930215812664840193]","[smart follow : <user>, i decided that i sh